# 📊 NSE + BSE Stock Analysis
### Darvas Box · Piotroski F-Score · Coffee Can Portfolio Screen

**Three quantitative scans across the full Indian equity universe:**
- 🔲 **Darvas Box** — technical momentum breakout (90-day OHLC via yfinance)
- 🧮 **Piotroski F-Score** — 9-point financial-strength score (yfinance annual statements)
- ☕ **Coffee Can** — Mukherjea's quality+growth filter (Revenue CAGR, ROCE, Debt, MCap)

Results are saved to **Google Drive** after every stock — safe to stop and re-run (auto-resumes).


In [ ]:
# Install required libraries (run once per Colab session)
!pip install nsepython yfinance openpyxl pandas tqdm -q
print("✅ Libraries ready")


## 💾 Mount Google Drive
Results are saved here so they survive session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")


## ⚙️ Configuration
Edit the settings below, then run all remaining cells.

In [ ]:
# @title ⚙️ Settings { run: "auto" }
import os
from pathlib import Path
from datetime import datetime

# ── What to analyse ───────────────────────────────────────────────────────────
MODE = "all-nse-bse"  # @param ["all-nse-bse", "excel"]
# "all-nse-bse" → full NSE (2364) + BSE-only (317) universe  [~2.8 h]
# "excel"       → your uploaded watchlist Excel file only     [~20 min]

# ── Excel watchlist (only needed when MODE == "excel") ────────────────────────
# Upload your Stock_List_NSE_BSE.xlsx using the Files panel on the left,
# then set the path here:
EXCEL_PATH = "/content/Stock_List_NSE_BSE.xlsx"  # @param {type:"string"}

# ── Output folder on Google Drive ─────────────────────────────────────────────
OUTPUT_FOLDER = "/content/drive/MyDrive/StockAnalysis"  # @param {type:"string"}

# ── Scan parameters ───────────────────────────────────────────────────────────
DARVAS_CONFIRM  = 3    # @param {type:"integer"}  confirmation days for Darvas Box
DELAY_BETWEEN   = 1.5  # @param {type:"number"}   seconds between stocks (rate-limit buffer)

# ── Create output directory ───────────────────────────────────────────────────
OUTPUT_DIR = Path(OUTPUT_FOLDER)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

today_str = datetime.today().strftime('%Y%m%d')
if MODE == "all-nse-bse":
    OUT_CSV = OUTPUT_DIR / f"full_analysis_{today_str}.csv"
else:
    OUT_CSV = OUTPUT_DIR / f"analysis_results_{today_str}.csv"

print(f"Mode        : {MODE}")
print(f"Output CSV  : {OUT_CSV}")
print(f"Darvas conf : {DARVAS_CONFIRM} days")
print(f"Delay       : {DELAY_BETWEEN}s / stock")


## 📦 Imports & Helpers

In [ ]:
import json, sys, time, warnings
from datetime import datetime, timedelta

import pandas as pd
import nsepython as nspy
import yfinance as yf
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print("✅ Imports OK — nsepython", nspy.__version__)


## 🗂️ Symbol List Builders

In [ ]:
def get_symbols_from_excel(path: str) -> list:
    """Load NSE symbols from the uploaded Excel watchlist."""
    df = pd.read_excel(path)
    syms = df["NSE Symbol"].dropna().str.strip().str.upper().unique().tolist()
    print(f"  Loaded {len(syms)} symbols from Excel")
    return [(s, "NS", "NSE") for s in syms]


def get_all_nse_bse_symbols() -> list:
    """
    Fetch the live NSE equity list + BSE-only EQ stocks.
    Returns list of (symbol, yf_suffix, exchange) tuples.
    NSE: yfinance SYMBOL.NS  |  BSE-only: yfinance SYMBOL.BO
    """
    print("  Fetching NSE equity symbols...", end="  ")
    nse_syms = [s.upper() for s in nspy.nse_eq_symbols()]
    print(f"{len(nse_syms)} found")

    print("  Fetching BSE bhavcopy...", end="  ")
    date_str = datetime.today().strftime("%d%m%Y")
    try:
        bse_bhav = nspy.get_bhavcopy(date_str)
        bse_bhav.columns = [c.strip() for c in bse_bhav.columns]
        bse_eq = (bse_bhav[bse_bhav["SERIES"].str.strip() == "EQ"]["SYMBOL"]
                  .str.strip().str.upper().unique().tolist())
    except Exception as e:
        print(f"FAILED ({e}) — using NSE only")
        bse_eq = []

    nse_set  = set(nse_syms)
    bse_only = [s for s in bse_eq if s not in nse_set]
    print(f"{len(bse_eq)} BSE EQ  →  {len(bse_only)} BSE-only after dedup")

    result  = [(s, "NS", "NSE") for s in nse_syms]
    result += [(s, "BO", "BSE") for s in bse_only]
    print(f"  Total unique symbols: {len(result)}")
    return result


## 🔧 yfinance Helpers

In [ ]:
def _yf_ticker(symbol: str, suffix: str = "NS"):
    return yf.Ticker(f"{symbol}.{suffix}")


def _yf_financials(ticker):
    """
    Fetch income statement, balance sheet, cash flow.
    Handles both old (financials/cashflow) and new (income_stmt/cash_flow) yfinance APIs.

    IMPORTANT: never use 'or' between DataFrames — raises
    'The truth value of a DataFrame is ambiguous'.
    Always use explicit None / .empty checks.
    """
    inc = getattr(ticker, "income_stmt", None)
    if inc is None or (hasattr(inc, "empty") and inc.empty):
        inc = getattr(ticker, "financials", None)

    bal = getattr(ticker, "balance_sheet", None)

    cf = getattr(ticker, "cash_flow", None)
    if cf is None or (hasattr(cf, "empty") and cf.empty):
        cf = getattr(ticker, "cashflow", None)

    return inc, bal, cf


def _row(df, *row_names, col: int = 0):
    """Safely read one scalar from a yfinance financial DataFrame."""
    if df is None or not hasattr(df, "empty") or df.empty:
        return None
    for name in row_names:
        if name in df.index:
            try:
                val = df.loc[name].iloc[col]
                return float(val) if pd.notna(val) else None
            except (IndexError, TypeError, ValueError):
                pass
    return None


def _series(df, *rows):
    """Return a row as a list of floats (most-recent first), skipping NaN."""
    if df is None or not hasattr(df, "empty") or df.empty:
        return []
    for name in rows:
        if name in df.index:
            return [float(v) for v in df.loc[name].dropna() if pd.notna(v)]
    return []

print("✅ yfinance helpers defined")


## 📡 Live Quote (nsepython / yfinance)

In [ ]:
def fetch_live_quote(symbol: str, exchange: str = "NSE") -> dict:
    """
    NSE stocks  → nsepython.nse_eq()   (richer metadata)
    BSE-only    → yfinance fast_info   (fallback)
    """
    if exchange == "NSE":
        try:
            data = nspy.nse_eq(symbol)
            pi   = data.get("priceInfo", {})
            md_  = data.get("metadata",  {})
            return {
                "ltp":         pi.get("lastPrice"),
                "prev_close":  pi.get("previousClose") or pi.get("prevClose"),
                "pct_change":  pi.get("pChange"),
                "day_high":    (pi.get("intraDayHighLow") or {}).get("max") or pi.get("dayHigh"),
                "day_low":     (pi.get("intraDayHighLow") or {}).get("min") or pi.get("dayLow"),
                "week52_high": (pi.get("weekHighLow") or {}).get("max"),
                "week52_low":  (pi.get("weekHighLow") or {}).get("min"),
                "vwap":        pi.get("vwap"),
                "company":     md_.get("companyName", ""),
                "industry":    md_.get("industry", ""),
            }
        except Exception as e:
            return {"error_quote": str(e)[:80]}
    else:
        try:
            fi = yf.Ticker(f"{symbol}.BO").fast_info
            return {
                "ltp":         getattr(fi, "last_price",     None),
                "prev_close":  getattr(fi, "previous_close", None),
                "pct_change":  None,
                "day_high":    getattr(fi, "day_high",       None),
                "day_low":     getattr(fi, "day_low",        None),
                "week52_high": getattr(fi, "year_high",      None),
                "week52_low":  getattr(fi, "year_low",       None),
                "vwap":        None,
                "company":     "",
                "industry":    "",
            }
        except Exception as e:
            return {"error_quote": str(e)[:80]}

print("✅ Live quote function defined")


## 🔲 Darvas Box Scan

In [ ]:
def fetch_ohlc_yf(symbol: str, suffix: str = "NS", days: int = 90) -> pd.DataFrame:
    """Fetch 90-day OHLC history from yfinance (SYMBOL.NS or SYMBOL.BO)."""
    try:
        df = _yf_ticker(symbol, suffix).history(period=f"{days}d")
        return df.reset_index() if not df.empty else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def compute_darvas_box(df: pd.DataFrame, confirm: int = DARVAS_CONFIRM) -> dict:
    """
    Darvas Box detection (Nicolas Darvas, 1960).

    Box Top    : most recent high not exceeded for `confirm` consecutive days
    Box Bottom : lowest low after box-top day, not undercut for `confirm` days
    Signal     : BREAKOUT_BUY / IN_BOX / BREAKDOWN_SELL / NO_BOX

    CRITICAL: box boundaries use history[:-1] (ALL bars EXCEPT the current bar).
    Including the current bar in boundary calculation makes BREAKDOWN_SELL
    impossible — the bar's own low would always pull box_bottom below its close.
    """
    def find_col(df, candidates):
        for c in candidates:
            m = next((col for col in df.columns if c.upper() in col.upper()), None)
            if m: return m
        return None

    h_col = find_col(df, ["High", "CH_TRADE_HIGH_PRICE"])
    l_col = find_col(df, ["Low",  "CH_TRADE_LOW_PRICE"])
    c_col = find_col(df, ["Close","CH_CLOSING_PRICE"])

    if not all([h_col, l_col, c_col]) or len(df) < confirm + 5:
        return {"signal": "INSUFFICIENT_DATA", "box_top": None, "box_bottom": None}

    all_highs  = pd.to_numeric(df[h_col], errors="coerce").fillna(0).tolist()
    all_lows   = pd.to_numeric(df[l_col], errors="coerce").fillna(0).tolist()
    all_closes = pd.to_numeric(df[c_col], errors="coerce").fillna(0).tolist()

    # Split: history for box formation | last bar for signal
    current = all_closes[-1]
    highs   = all_highs[:-1]
    lows    = all_lows[:-1]
    n       = len(highs)

    # Step 1 — confirmed box top
    box_top_idx = box_top = None
    for i in range(n - confirm - 1, -1, -1):
        c = highs[i]
        if c == 0: continue
        w = highs[i+1 : i+1+confirm]
        if len(w) == confirm and all(h < c for h in w):
            box_top_idx, box_top = i, c
            break

    if box_top is None:
        return {"signal": "NO_BOX", "box_top": None, "box_bottom": None}

    # Step 2 — confirmed box bottom (historical segment only)
    segment    = lows[box_top_idx:]
    box_bottom = None
    for i in range(len(segment) - confirm):
        c = segment[i]
        if c == 0: continue
        w = segment[i+1 : i+1+confirm]
        if len(w) == confirm and all(l > c for l in w):
            box_bottom = c
            break
    if box_bottom is None:
        valid = [l for l in segment if l > 0]
        box_bottom = min(valid) if valid else None

    if box_bottom is None:
        return {"signal": "NO_BOX", "box_top": box_top, "box_bottom": None}

    # Step 3 — classify today's close against the frozen box
    signal = ("BREAKOUT_BUY"   if current > box_top   else
              "BREAKDOWN_SELL" if current < box_bottom else "IN_BOX")
    box_range = box_top - box_bottom
    pos       = ((current - box_bottom) / box_range * 100) if box_range else 0

    return {
        "signal":              signal,
        "box_top":             round(box_top,    2),
        "box_bottom":          round(box_bottom, 2),
        "current_price":       round(current,    2),
        "box_range":           round(box_range,  2),
        "position_in_box_pct": round(pos,        1),
    }

print("✅ Darvas Box functions defined")


## 🧮 Piotroski F-Score

In [ ]:
def compute_piotroski(symbol: str, suffix: str = "NS", ticker=None) -> dict:
    """
    Piotroski F-Score (0–9): binary scores across profitability,
    leverage/liquidity, and operating efficiency.
    Score ≥ 7 → Strong  |  4–6 → Moderate  |  ≤ 3 → Weak
    """
    try:
        if ticker is None:
            ticker = _yf_ticker(symbol, suffix)
        inc, bal, cf = _yf_financials(ticker)
    except Exception as e:
        return {"f_score": None, "f_interpretation": f"error: {e}"[:60]}

    if inc is None or (hasattr(inc, "empty") and inc.empty):
        return {"f_score": None, "f_interpretation": "no income data"}

    s = {}

    # Profitability
    ni0  = _row(inc, "Net Income",    col=0)
    ta0  = _row(bal, "Total Assets",  col=0)
    ni1  = _row(inc, "Net Income",    col=1)
    ta1  = _row(bal, "Total Assets",  col=1)
    roa0 = (ni0/ta0) if (ni0 and ta0) else None
    roa1 = (ni1/ta1) if (ni1 and ta1) else None

    s["F1"] = 1 if (roa0 and roa0 > 0) else 0
    ocf0 = _row(cf, "Operating Cash Flow",
                    "Total Cash From Operating Activities", col=0)
    s["F2"] = 1 if (ocf0 and ocf0 > 0) else 0
    s["F3"] = 1 if (roa0 and roa1 and roa0 > roa1) else 0
    s["F4"] = 1 if (ocf0 and ta0 and roa0 is not None
                    and (ocf0/ta0) > roa0) else 0

    # Leverage & Liquidity
    ltd0 = _row(bal, "Long Term Debt", col=0) or 0
    ltd1 = _row(bal, "Long Term Debt", col=1) or 0
    lev0 = (ltd0/ta0) if ta0 else None
    lev1 = (ltd1/ta1) if ta1 else None
    s["F5"] = 1 if (lev0 is not None and lev1 is not None
                    and lev0 < lev1) else 0

    ca0 = _row(bal, "Current Assets","Total Current Assets",       col=0)
    cl0 = _row(bal, "Current Liabilities","Total Current Liabilities", col=0)
    ca1 = _row(bal, "Current Assets","Total Current Assets",       col=1)
    cl1 = _row(bal, "Current Liabilities","Total Current Liabilities", col=1)
    cr0 = (ca0/cl0) if (ca0 and cl0) else None
    cr1 = (ca1/cl1) if (ca1 and cl1) else None
    s["F6"] = 1 if (cr0 and cr1 and cr0 > cr1) else 0

    sh0 = _row(bal, "Share Issued", col=0)
    sh1 = _row(bal, "Share Issued", col=1)
    s["F7"] = 1 if (sh0 and sh1 and sh0 <= sh1) else 1  # assume OK if missing

    # Operating Efficiency
    rev0 = _row(inc, "Total Revenue", col=0)
    gp0  = _row(inc, "Gross Profit",  col=0)
    rev1 = _row(inc, "Total Revenue", col=1)
    gp1  = _row(inc, "Gross Profit",  col=1)
    gm0  = (gp0/rev0) if (gp0 and rev0) else None
    gm1  = (gp1/rev1) if (gp1 and rev1) else None
    s["F8"] = 1 if (gm0 and gm1 and gm0 > gm1) else 0
    at0  = (rev0/ta0) if (rev0 and ta0) else None
    at1  = (rev1/ta1) if (rev1 and ta1) else None
    s["F9"] = 1 if (at0 and at1 and at0 > at1) else 0

    total  = sum(s.values())
    interp = ("Strong" if total >= 7 else "Moderate" if total >= 4 else "Weak")
    return {
        "f_score":          total,
        "f_interpretation": interp,
        "f_roa_pct":        round(roa0*100, 2) if roa0 else None,
        "f_ocf_cr":         round(ocf0/1e7, 2) if ocf0 else None,
        "f_components":     json.dumps(s),
    }

print("✅ Piotroski F-Score function defined")


## ☕ Coffee Can Screen

In [ ]:
def compute_coffee_can(symbol: str, suffix: str = "NS", ticker=None) -> dict:
    """
    Coffee Can Portfolio screen (Saurabh Mukherjea, Marcellus Investment Managers).
    All 5 must pass to qualify:
      C1  Revenue CAGR > 10%
      C2  Average ROCE > 15%
      C3  Debt/Equity < 1
      C4  Market Cap ≥ ₹500 Cr
      C5  No loss-making year in available history
    """
    try:
        if ticker is None:
            ticker = _yf_ticker(symbol, suffix)
        inc, bal, _ = _yf_financials(ticker)
        info = ticker.info or {}
    except Exception as e:
        return {"cc_qualifies": None, "cc_score": None, "cc_note": str(e)[:60]}

    if inc is None or (hasattr(inc, "empty") and inc.empty):
        return {"cc_qualifies": None, "cc_score": None, "cc_note": "no income data"}

    c = {}

    # C1 — Revenue CAGR > 10%
    revs = _series(inc, "Total Revenue")
    cagr = None
    if len(revs) >= 2:
        years = len(revs) - 1
        cagr  = ((revs[0]/revs[-1])**(1/years) - 1)*100 if revs[-1] > 0 else None
    c["C1"] = 1 if (cagr and cagr > 10) else 0

    # C2 — Average ROCE > 15%
    ebit_s = _series(inc, "EBIT", "Operating Income", "Ebit")
    ta_s   = _series(bal, "Total Assets")
    cl_s   = _series(bal, "Current Liabilities", "Total Current Liabilities")
    roce_list = []
    for i in range(min(len(ebit_s), len(ta_s), len(cl_s))):
        ce = ta_s[i] - cl_s[i]
        if ce > 0: roce_list.append(ebit_s[i]/ce*100)
    avg_roce = (sum(roce_list)/len(roce_list)) if roce_list else None
    c["C2"] = 1 if (avg_roce and avg_roce > 15) else 0

    # C3 — Debt/Equity < 1
    de = None
    de_raw = info.get("debtToEquity")
    if de_raw is not None:
        de = de_raw/100 if de_raw > 10 else de_raw
        c["C3"] = 1 if de < 1 else 0
    else:
        ltd_s = _series(bal, "Long Term Debt")
        eq_s  = _series(bal, "Stockholders Equity","Total Stockholder Equity",
                        "Total Equity Gross Minority Interest")
        if ltd_s and eq_s and eq_s[0] != 0:
            de = ltd_s[0]/eq_s[0]
            c["C3"] = 1 if de < 1 else 0
        else:
            c["C3"] = 0

    # C4 — Market Cap ≥ ₹500 Cr
    mcap    = info.get("marketCap")
    mcap_cr = (mcap/1e7) if mcap else None
    c["C4"] = 1 if (mcap_cr and mcap_cr >= 500) else 0

    # C5 — No loss year
    ni_s = _series(inc, "Net Income")
    c["C5"] = 1 if (ni_s and all(n > 0 for n in ni_s)) else 0

    total     = sum(c.values())
    qualifies = (total == len(c))
    return {
        "cc_qualifies":   qualifies,
        "cc_score":       f"{total}/{len(c)}",
        "cc_rev_cagr":    round(cagr,     2) if cagr     else None,
        "cc_roce_avg":    round(avg_roce, 2) if avg_roce else None,
        "cc_debt_equity": round(de,       2) if de       else None,
        "cc_mcap_cr":     round(mcap_cr,  2) if mcap_cr  else None,
        "cc_loss_years":  sum(1 for n in ni_s if n <= 0) if ni_s else None,
    }

print("✅ Coffee Can function defined")


## 🚀 Batch Runner

In [ ]:
def run_batch(symbol_list: list, output_path: Path) -> Path:
    """
    Run all three scans for each (symbol, yf_suffix, exchange) tuple.
    Appends to CSV after every stock — safe to stop and resume.
    Re-running automatically skips already-completed symbols.
    """
    # Load done symbols for resume
    done = set()
    if output_path.exists():
        try:
            done = set(pd.read_csv(output_path)["symbol"].str.upper())
            print(f"  Resuming — {len(done)} symbols already done, skipping them.\n")
        except Exception:
            pass

    header_written = output_path.exists()
    remaining = [(s,sf,ex) for s,sf,ex in symbol_list if s.upper() not in done]

    print(f"  Symbols to process: {len(remaining)} / {len(symbol_list)}")
    est_h = len(remaining) * DELAY_BETWEEN * 3 / 3600
    print(f"  Estimated time    : {est_h:.1f} h  (Colab will timeout after 90 min —")
    print(f"                       just re-run this cell to resume from checkpoint)\n")

    for sym, suffix, exchange in tqdm(remaining, desc="Analysing stocks", unit="stock"):
        row = {"symbol": sym, "exchange": exchange,
               "timestamp": datetime.now().isoformat()}

        # 1. Live quote
        q = fetch_live_quote(sym, exchange=exchange)
        row.update(q)

        # 2. Darvas Box
        ohlc = fetch_ohlc_yf(sym, suffix=suffix, days=90)
        darv = compute_darvas_box(ohlc)
        row["darvas_signal"]     = darv.get("signal")
        row["darvas_box_top"]    = darv.get("box_top")
        row["darvas_box_bottom"] = darv.get("box_bottom")
        row["darvas_pos_pct"]    = darv.get("position_in_box_pct")

        # 3 & 4. Piotroski + Coffee Can (share one Ticker call)
        try:
            ticker = _yf_ticker(sym, suffix)
            piotr  = compute_piotroski(sym,  suffix=suffix, ticker=ticker)
            coffee = compute_coffee_can(sym, suffix=suffix, ticker=ticker)
            row.update(piotr)
            row.update(coffee)
        except Exception as e:
            row["yf_error"] = str(e)[:80]

        # Append to CSV
        pd.DataFrame([row]).to_csv(
            output_path, mode="a", header=not header_written, index=False
        )
        header_written = True
        time.sleep(DELAY_BETWEEN)

    print(f"\n✅ Done — results saved to {output_path}")
    return output_path

print("✅ Batch runner defined")


## 📊 Summary & Results

In [ ]:
from IPython.display import display, HTML
import pandas as pd

def show_summary(output_path: Path):
    """Display a styled HTML summary of the analysis results."""
    try:
        df = pd.read_csv(output_path)
    except FileNotFoundError:
        print("No results file found yet.")
        return

    total = len(df)
    display(HTML(f"<h2>📊 Analysis Summary — {total} stocks</h2>"))

    # Exchange breakdown
    if "exchange" in df.columns:
        exch_html = df["exchange"].value_counts().to_frame().to_html()
        display(HTML(f"<h3>Exchange Breakdown</h3>{exch_html}"))

    # Darvas signals
    if "darvas_signal" in df.columns:
        sig_html = df["darvas_signal"].value_counts().to_frame().to_html()
        display(HTML(f"<h3>🔲 Darvas Box Signals</h3>{sig_html}"))

        breakouts = df[df["darvas_signal"] == "BREAKOUT_BUY"].copy()
        if not breakouts.empty:
            breakouts = breakouts.sort_values("darvas_pos_pct", ascending=False)
            cols = ["symbol","exchange","ltp","pct_change","darvas_box_top","darvas_box_bottom","darvas_pos_pct"]
            cols = [c for c in cols if c in breakouts.columns]
            display(HTML(f"<h3>✅ BREAKOUT_BUY stocks ({len(breakouts)})</h3>"
                        + breakouts[cols].to_html(index=False)))

    # Piotroski Strong
    if "f_score" in df.columns:
        df["f_score"] = pd.to_numeric(df["f_score"], errors="coerce")
        strong = df[df["f_score"] >= 7].sort_values("f_score", ascending=False)
        cols = ["symbol","exchange","f_score","f_interpretation","ltp","pct_change"]
        cols = [c for c in cols if c in strong.columns]
        display(HTML(f"<h3>🧮 Piotroski STRONG — F ≥ 7  ({len(strong)} stocks)</h3>"
                    + strong[cols].head(50).to_html(index=False)))

    # Coffee Can
    if "cc_qualifies" in df.columns:
        qual = df[df["cc_qualifies"] == True].copy()
        cols = ["symbol","exchange","cc_score","cc_rev_cagr","cc_roce_avg","cc_mcap_cr","ltp"]
        cols = [c for c in cols if c in qual.columns]
        display(HTML(f"<h3>☕ Coffee Can QUALIFIES  ({len(qual)} stocks)</h3>"
                    + qual[cols].to_html(index=False)))

    # Triple Hit
    if all(c in df.columns for c in ["darvas_signal","f_score","cc_qualifies"]):
        triple = df[
            (df["darvas_signal"] == "BREAKOUT_BUY") &
            (df["f_score"] >= 7) &
            (df["cc_qualifies"] == True)
        ]
        cols = ["symbol","exchange","darvas_signal","f_score","cc_score","ltp","pct_change"]
        cols = [c for c in cols if c in triple.columns]
        html = (triple[cols].to_html(index=False)
                if not triple.empty else "<p><em>None found yet.</em></p>")
        display(HTML(f"<h3>⭐ TRIPLE HIT: Breakout + F≥7 + Coffee Can  ({len(triple)})</h3>" + html))

    # Download link for the CSV
    try:
        from google.colab import files
        print(f"\nResults file: {output_path}")
        print("Use Files panel (left sidebar) to download the CSV from Google Drive.")
    except Exception:
        pass

print("✅ Summary function defined")


---
## ▶️ Run the Analysis

**Step 1** — Build symbol list  
**Step 2** — Run batch (re-run this cell anytime to resume after a timeout)  
**Step 3** — View summary


In [ ]:
# Step 1 — Build symbol list
if MODE == "all-nse-bse":
    symbol_list = get_all_nse_bse_symbols()
else:
    symbol_list = get_symbols_from_excel(EXCEL_PATH)

print(f"\nReady to analyse {len(symbol_list)} symbols → {OUT_CSV}")


In [ ]:
# Step 2 — Run batch
# Re-run this cell after a Colab timeout — it resumes automatically from checkpoint.
run_batch(symbol_list, OUT_CSV)


In [ ]:
# Step 3 — View summary (run anytime, even while Step 2 is still going)
show_summary(OUT_CSV)


## 💾 Export Results to Excel

In [ ]:
# Convert the CSV results to a formatted Excel file with multiple sheets
def export_to_excel(csv_path: Path):
    df = pd.read_csv(csv_path)
    df["f_score"] = pd.to_numeric(df["f_score"], errors="coerce")

    xlsx_path = csv_path.with_suffix(".xlsx")
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        # All results
        df.to_excel(writer, sheet_name="All Stocks", index=False)

        # Breakouts
        if "darvas_signal" in df.columns:
            (df[df["darvas_signal"] == "BREAKOUT_BUY"]
             .sort_values("darvas_pos_pct", ascending=False)
             .to_excel(writer, sheet_name="Darvas Breakouts", index=False))

        # Piotroski strong
        if "f_score" in df.columns:
            (df[df["f_score"] >= 7]
             .sort_values("f_score", ascending=False)
             .to_excel(writer, sheet_name="Piotroski Strong", index=False))

        # Coffee Can
        if "cc_qualifies" in df.columns:
            (df[df["cc_qualifies"] == True]
             .to_excel(writer, sheet_name="Coffee Can", index=False))

        # Triple hits
        if all(c in df.columns for c in ["darvas_signal","f_score","cc_qualifies"]):
            (df[(df["darvas_signal"]=="BREAKOUT_BUY") &
                (df["f_score"] >= 7) &
                (df["cc_qualifies"] == True)]
             .to_excel(writer, sheet_name="Triple Hits", index=False))

    print(f"✅ Excel saved → {xlsx_path}")
    return xlsx_path

export_to_excel(OUT_CSV)
